<div style="background:linear-gradient(90deg,#0366d6,#1abc9c); color:white; padding:20px 32px; border-radius:8px; width:97%;">

## 📖 `read_file` 中 `offset` 与 `limit` 的作用

</div>

简单说,这两个参数实现的是 **分页读取(pagination)** —— 像翻书一样,**跳过前 N 行,读后面 M 行**。

## 🎯 参数含义

| 参数 | 默认值 | 作用 |
|---|---|---|
| `offset` | `0` | **从第几行开始读**(skip 前面 N 行)|
| `limit` | `2000` | **最多读多少行** |

## 🔧 代码做了啥(三行核心逻辑)

```python
lines = content.splitlines()                  # 把文件按行切开
start_idx = offset                            # 起点
end_idx = min(start_idx + limit, len(lines))  # 终点(不超过文件末尾)
```

然后返回 `lines[start_idx : end_idx]`。

## 📋 举例

假设文件有 **5000 行**:

| 调用 | 读取的行 |
|---|---|
| `read_file("foo.py")` | 1–2000(默认值,前 2000 行) |
| `read_file("foo.py", offset=2000)` | 2001–4000 |
| `read_file("foo.py", offset=2000, limit=500)` | 2001–2500 |
| `read_file("foo.py", offset=4500)` | 4501–5000(只剩 500 行) |
| `read_file("foo.py", offset=10000)` | ⚠️ 返回错误:`offset 10000 exceeds file length (5000 lines)` |

## 💡 为什么要这套机制?

> **省 token + 防爆 context**。

LLM 的 context 是有限的(几万到几十万 token)。如果一个文件有 50000 行,一次性全读进来会:

- 💸 **token 成本暴涨**
- 🧊 **关键信息被淹没**(context dilution)
- 💥 **可能直接超出 context window**

所以工具设计成 **分页**:

1. 🚀 LLM 先 `read_file(path)` → 拿到前 2000 行(默认 limit)
2. 👀 看到"还有内容、感兴趣"的部分 → 再调 `read_file(path, offset=2000)`
3. ⏭️ 不感兴趣的部分 → **直接跳过,不读**

## 🔍 你大概率眼熟这套设计

<div style="background:#e8f4fd; padding:12px 24px; border-left:4px solid #0366d6; border-radius:4px; width:97%;">

**Claude Code 的 `Read` 工具签名就是**:

```python
Read(file_path, offset, limit)
```

这是 **2024–2026 年 agent 工具设计的标准模式** —— 任何"读大资源"的工具(文件、长 trace、网页内容)都该带 `offset` + `limit`,让 LLM **自己决定读多少**。

</div>

## ✨ 一句话总结

> 🔑 **`offset` = "跳到哪",`limit` = "读多少"**。
>
> 合起来就是给 LLM 一个 **"按需翻页"** 的能力,而不是强迫它一口气吞掉整个文件。